In [17]:
# Upload CSV file
from google.colab import files
uploaded = files.upload()

Saving cleaned_transactions.csv to cleaned_transactions (2).csv


In [18]:
#  Load data and set up SQLite
import pandas as pd
import sqlite3

# Load CSV into pandas dataframe
df = pd.read_csv('cleaned_transactions.csv')

# Show shape and first 5 rows to verify
print("Shape:", df.shape)
print("\nColumns:", df.columns.tolist())
print("\nFirst 5 rows:")
df.head()

Shape: (30, 19)

Columns: ['transaction_id', 'transaction_date', 'merchant_name', 'raw_amount', 'currency', 'status', 'risk_score', 'gateway_region', 'user_id', 'payment_method', 'clean_merchant', 'clean_date', 'clean_status', 'clean_risk_score', 'clean_region', 'amount_usd', 'merchant_category', 'high_value_flag', 'high_risk_flag']

First 5 rows:


,transaction_id,transaction_date,merchant_name,raw_amount,currency,status,risk_score,gateway_region,user_id,payment_method,clean_merchant,clean_date,clean_status,clean_risk_score,clean_region,amount_usd,merchant_category,high_value_flag,high_risk_flag
0,T001,01-03-2026,alpha mart,420000,INR,captured,score:62,APAC,U001,UPI,Alpha Mart,2026-03-01,captured,62.0,APAC,35294117.65,Unknown,1,0
1,T002,01-03-2026,ALPHA MART,210000,INR,Captured,55,NaN,U002,Card,Alpha Mart,2026-03-01,captured,55.0,NaN,17647058.82,Unknown,0,0
2,T003,01-03-2026,BETA STORES,510000,INR,CAPTURED,71,APAC,U003,NetBanking,Beta Stores,2026-03-01,captured,71.0,APAC,42857142.86,Unknown,1,1
3,T004,02-03-2026,Beta Stores,160000,INR,failed e05 timeout,68,apac,U004,Card,Beta Stores,2026-03-02,failed_timeout,68.0,APAC,13445378.15,Unknown,1,0
4,T005,02-03-2026,Alpha Mart,390000,INR,CAPTURED,58,NaN,U001,UPI,Alpha Mart,2026-03-02,captured,58.0,NaN,32773109.24,Unknown,0,0


In [19]:
#  Create in-memory SQLite database
conn = sqlite3.connect(':memory:')

# Load dataframe into SQLite as a table called 'transactions'
df.to_sql('transactions', conn, index=False, if_exists='replace')

print("✅ Database ready! Table 'transactions' created.")
print(f"Total rows loaded: {len(df)}")

✅ Database ready! Table 'transactions' created.
Total rows loaded: 30


In [20]:
#Check exact column names
query = "PRAGMA table_info(transactions);"
result = pd.read_sql_query(query, conn)
print(result[['name', 'type']])

                 name     type
0      transaction_id     TEXT
1    transaction_date     TEXT
2       merchant_name     TEXT
3          raw_amount  INTEGER
4            currency     TEXT
5              status     TEXT
6          risk_score     TEXT
7      gateway_region     TEXT
8             user_id     TEXT
9      payment_method     TEXT
10     clean_merchant     TEXT
11         clean_date     TEXT
12       clean_status     TEXT
13   clean_risk_score     REAL
14       clean_region     TEXT
15         amount_usd     REAL
16  merchant_category     TEXT
17    high_value_flag  INTEGER
18     high_risk_flag  INTEGER


In [21]:
### Q1: Count transactions by status
q1 = """
SELECT status,
       COUNT(*) AS transaction_count
FROM transactions
GROUP BY status
ORDER BY transaction_count DESC;
"""

result_q1 = pd.read_sql_query(q1, conn)
print("Q1 - Count by Status:")
print(result_q1.to_string(index=False))

Q1 - Count by Status:
              status  transaction_count
           captured                   7
           Captured                   5
            captured                  4
  Failed E05 Timeout                  4
         chargeback                   3
            CAPTURED                  3
  failed E05 timeout                  1
          chargeback                  1
  FAILED e05 TIMEOUT                  1
 failed e05 timeout                   1


In [22]:
### Q2: Total captured GMV by merchant
q2 = """
SELECT merchant_name,
       ROUND(SUM(amount_usd), 2) AS total_gmv
FROM transactions
WHERE status = 'captured'
GROUP BY merchant_name
ORDER BY total_gmv DESC;
"""

result_q2 = pd.read_sql_query(q2, conn)
print("Q2 - Total GMV by Merchant:")
print(result_q2.to_string(index=False))

Q2 - Total GMV by Merchant:
merchant_name   total_gmv
  Beta Stores 51260504.20
  alpha mart  35294117.65
   Alpha Mart 21428571.43


In [23]:
### Q3: Top 10 merchants by captured GMV
q3 = """
SELECT merchant_name,
       ROUND(SUM(amount_usd), 2) AS total_gmv,
       COUNT(*) AS transaction_count
FROM transactions
WHERE status = 'captured'
GROUP BY merchant_name
ORDER BY total_gmv DESC
LIMIT 10;
"""

result_q3 = pd.read_sql_query(q3, conn)
print("Q3 - Top 10 Merchants:")
print(result_q3.to_string(index=False))

Q3 - Top 10 Merchants:
merchant_name   total_gmv  transaction_count
  Beta Stores 51260504.20                  1
  alpha mart  35294117.65                  1
   Alpha Mart 21428571.43                  2


In [24]:
### Q4: Daily GMV and successful transaction count
q4 = """
SELECT transaction_date,
       ROUND(SUM(amount_usd), 2) AS daily_gmv,
       COUNT(*) AS successful_count
FROM transactions
WHERE status = 'captured'
GROUP BY transaction_date
ORDER BY transaction_date ASC;
"""

result_q4 = pd.read_sql_query(q4, conn)
print("Q4 - Daily GMV:")
print(result_q4.to_string(index=False))

Q4 - Daily GMV:
transaction_date   daily_gmv  successful_count
      01-03-2026 35294117.65                 1
      03-03-2026 61764705.88                 2
      04-03-2026 10924369.75                 1


In [25]:
### Q5: Merchants with chargeback ratio above 1%
q5 = """
SELECT merchant_name,
       COUNT(*) AS total_transactions,
       SUM(CASE WHEN status = 'chargeback' THEN 1 ELSE 0 END) AS chargeback_count,
       ROUND(
         100.0 * SUM(CASE WHEN status = 'chargeback' THEN 1 ELSE 0 END) / COUNT(*),
         2
       ) AS chargeback_pct
FROM transactions
GROUP BY merchant_name
HAVING chargeback_pct > 1
ORDER BY chargeback_pct DESC;
"""

result_q5 = pd.read_sql_query(q5, conn)
print("Q5 - High Chargeback Merchants:")
print(result_q5.to_string(index=False))

Q5 - High Chargeback Merchants:
merchant_name  total_transactions  chargeback_count  chargeback_pct
  Alpha  Mart                   3                 1           33.33


In [26]:
### Q6: Regions with avg risk score > 50 and 20+ transactions
q6 = """
SELECT gateway_region,
       ROUND(AVG(risk_score), 2) AS avg_risk_score,
       COUNT(*) AS total_transactions
FROM transactions
WHERE risk_score IS NOT NULL
  AND gateway_region IS NOT NULL
GROUP BY gateway_region
HAVING avg_risk_score > 50
   AND total_transactions > 20
ORDER BY avg_risk_score DESC;
"""

result_q6 = pd.read_sql_query(q6, conn)
print("Q6 - High Risk Regions:")
print(result_q6.to_string(index=False))

Q6 - High Risk Regions:
Empty DataFrame
Columns: [gateway_region, avg_risk_score, total_transactions]
Index: []


In [27]:
### Q7: Users with 3+ failed/chargeback on same day
q7 = """
SELECT user_id,
       transaction_date,
       COUNT(*) AS fail_count
FROM transactions
WHERE status IN ('failed_timeout', 'chargeback')
GROUP BY user_id, transaction_date
HAVING fail_count >= 3
ORDER BY fail_count DESC;
"""

result_q7 = pd.read_sql_query(q7, conn)
print("Q7 - Suspicious Users:")
print(result_q7.to_string(index=False))

Q7 - Suspicious Users:
Empty DataFrame
Columns: [user_id, transaction_date, fail_count]
Index: []


In [28]:
### Q8: Chargeback summary by merchant
q8 = """
SELECT merchant_name,
       COUNT(*) AS chargeback_count,
       COUNT(DISTINCT user_id) AS unique_affected_users,
       ROUND(SUM(amount_usd), 2) AS chargeback_amount
FROM transactions
WHERE status = 'chargeback'
GROUP BY merchant_name
ORDER BY chargeback_amount DESC;
"""

result_q8 = pd.read_sql_query(q8, conn)
print("Q8 - Chargeback by Merchant:")
print(result_q8.to_string(index=False))

Q8 - Chargeback by Merchant:
merchant_name  chargeback_count  unique_affected_users  chargeback_amount
  Alpha  Mart                 1                      1        37815126.05


In [29]:
print(df.columns.tolist())


['transaction_id', 'transaction_date', 'merchant_name', 'raw_amount', 'currency', 'status', 'risk_score', 'gateway_region', 'user_id', 'payment_method', 'clean_merchant', 'clean_date', 'clean_status', 'clean_risk_score', 'clean_region', 'amount_usd', 'merchant_category', 'high_value_flag', 'high_risk_flag']


In [30]:
# Check actual values in clean_status column
print(df['clean_status'].unique())

['captured' 'failed_timeout' 'chargeback']


In [34]:
# CELL - Q1: Count transactions by status
q1 = """
SELECT clean_status,
       COUNT(*) AS transaction_count
FROM transactions
GROUP BY clean_status
ORDER BY transaction_count DESC;
"""

result_q1 = pd.read_sql_query(q1, conn)
print("Q1 - Count by Status:")
print(result_q1.to_string(index=False))

Q1 - Count by Status:
  clean_status  transaction_count
      captured                 19
failed_timeout                  7
    chargeback                  4


In [37]:
# CELL - Q2: Total captured GMV by merchant
q2 = """
SELECT clean_merchant,
       ROUND(SUM(amount_usd), 2) AS total_gmv
FROM transactions
WHERE clean_status = 'captured'
GROUP BY clean_merchant
ORDER BY total_gmv DESC;
"""

result_q2 = pd.read_sql_query(q2, conn)
print("Q2 - Total GMV by Merchant:")
print(result_q2.to_string(index=False))

Q2 - Total GMV by Merchant:
clean_merchant    total_gmv
   Beta Stores 234033613.45
    Alpha Mart 211344537.81
 Delta Travels     10300.00
   City Pharma      7407.40


In [38]:
# CELL - Q3: Top 10 merchants by captured GMV
q3 = """
SELECT clean_merchant,
       ROUND(SUM(amount_usd), 2) AS total_gmv,
       COUNT(*) AS transaction_count
FROM transactions
WHERE clean_status = 'captured'
GROUP BY clean_merchant
ORDER BY total_gmv DESC
LIMIT 10;
"""

result_q3 = pd.read_sql_query(q3, conn)
print("Q3 - Top 10 Merchants:")
print(result_q3.to_string(index=False))

Q3 - Top 10 Merchants:
clean_merchant    total_gmv  transaction_count
   Beta Stores 234033613.45                  7
    Alpha Mart 211344537.81                  8
 Delta Travels     10300.00                  2
   City Pharma      7407.40                  2


In [39]:
# CELL - Q4: Daily GMV and successful transaction count
q4 = """
SELECT clean_date,
       ROUND(SUM(amount_usd), 2) AS daily_gmv,
       COUNT(*) AS successful_count
FROM transactions
WHERE clean_status = 'captured'
GROUP BY clean_date
ORDER BY clean_date ASC;
"""

result_q4 = pd.read_sql_query(q4, conn)
print("Q4 - Daily GMV:")
print(result_q4.to_string(index=False))

Q4 - Daily GMV:
clean_date   daily_gmv  successful_count
2026-03-01 95810334.14                 5
2026-03-02 55885452.94                 3
2026-03-03 90338727.04                 4
2026-03-04 97478991.60                 4
2026-03-05 43697478.99                 1
2026-03-06 62184873.95                 2


In [40]:
# CELL - Q5: Merchants with chargeback ratio above 1%
q5 = """
SELECT clean_merchant,
       COUNT(*) AS total_transactions,
       SUM(CASE WHEN clean_status = 'chargeback' THEN 1 ELSE 0 END) AS chargeback_count,
       ROUND(
         100.0 * SUM(CASE WHEN clean_status = 'chargeback' THEN 1 ELSE 0 END) / COUNT(*),
         2
       ) AS chargeback_pct
FROM transactions
GROUP BY clean_merchant
HAVING chargeback_pct > 1
ORDER BY chargeback_pct DESC;
"""

result_q5 = pd.read_sql_query(q5, conn)
print("Q5 - High Chargeback Merchants:")
print(result_q5.to_string(index=False))

Q5 - High Chargeback Merchants:
clean_merchant  total_transactions  chargeback_count  chargeback_pct
      Eco Home                   2                 1           50.00
 Delta Travels                   4                 1           25.00
   Beta Stores                  11                 1            9.09
    Alpha Mart                  11                 1            9.09


In [60]:
# CELL - Q6: Regions with avg risk score > 50 and 20+ transactions
q6 ="""
SELECT clean_region,
       ROUND(AVG(clean_risk_score), 2) AS avg_risk_score,
       COUNT(*) AS total_transactions
FROM transactions
WHERE clean_risk_score IS NOT NULL
  AND clean_region IS NOT NULL
GROUP BY clean_region
HAVING avg_risk_score > 50
ORDER BY avg_risk_score DESC;
"""

result_q6 = pd.read_sql_query(q6, conn)
print("Q6 - High Risk Regions (adjusted for small dataset):")
print(result_q6.to_string(index=False))

Q6 - High Risk Regions:
clean_region  avg_risk_score  total_transactions
        APAC           67.54                  13


In [47]:
print(df['clean_status'].unique())

['captured' 'failed_timeout' 'chargeback']


In [61]:
# CELL - Q7: Users with 3+ failed/chargeback on same day
q7 = """
SELECT user_id,
       clean_date,
       COUNT(*) AS fail_count
FROM transactions
WHERE clean_status IN ('failed_timeout', 'chargeback')
GROUP BY user_id, clean_date
HAVING fail_count >= 3
ORDER BY fail_count DESC;
"""

result_q7 = pd.read_sql_query(q7, conn)
print("Q7 - Suspicious Users:")
print(result_q7.to_string(index=False))

Q7 - Suspicious Users:
user_id clean_date  fail_count
   U008 2026-03-05           4


In [62]:
# CELL - Q8: Chargeback summary by merchant
q8 = """
SELECT clean_merchant,
       COUNT(*) AS chargeback_count,
       COUNT(DISTINCT user_id) AS unique_affected_users,
       ROUND(SUM(amount_usd), 2) AS chargeback_amount
FROM transactions
WHERE clean_status = 'chargeback'
GROUP BY clean_merchant
ORDER BY chargeback_amount DESC;
"""

result_q8 = pd.read_sql_query(q8, conn)
print("Q8 - Chargeback by Merchant:")
print(result_q8.to_string(index=False))

Q8 - Chargeback by Merchant:
clean_merchant  chargeback_count  unique_affected_users  chargeback_amount
    Alpha Mart                 1                      1        37815126.05
   Beta Stores                 1                      1        12184873.95
      Eco Home                 1                      1            5648.15
 Delta Travels                 1                      1            2500.00


In [48]:
# DIAGNOSIS - Check actual data in clean_region and clean_risk_score
print("=" * 50)
print("UNIQUE REGIONS:")
print(df['clean_region'].unique())

print("\n" + "=" * 50)
print("REGION COUNT (how many rows per region):")
print(df['clean_region'].value_counts())

print("\n" + "=" * 50)
print("RISK SCORE DATA TYPE:")
print(df['clean_risk_score'].dtype)

print("\n" + "=" * 50)
print("RISK SCORE SAMPLE VALUES:")
print(df['clean_risk_score'].head(10))

print("\n" + "=" * 50)
print("TOTAL ROWS IN DATASET:")
print(len(df))

UNIQUE REGIONS:
['APAC' nan 'EU' 'US']

REGION COUNT (how many rows per region):
clean_region
APAC    13
EU       4
US       4
Name: count, dtype: int64

RISK SCORE DATA TYPE:
float64

RISK SCORE SAMPLE VALUES:
0    62.0
1    55.0
2    71.0
3    68.0
4    58.0
5    64.0
6    83.0
7    59.0
8    46.0
9    77.0
Name: clean_risk_score, dtype: float64

TOTAL ROWS IN DATASET:
30


In [49]:
# FIX - Convert clean_risk_score from text to number
df['clean_risk_score'] = pd.to_numeric(df['clean_risk_score'], errors='coerce')

print("Data type after fix:")
print(df['clean_risk_score'].dtype)

print("\nSample values after fix:")
print(df['clean_risk_score'].head(10))

Data type after fix:
float64

Sample values after fix:
0    62.0
1    55.0
2    71.0
3    68.0
4    58.0
5    64.0
6    83.0
7    59.0
8    46.0
9    77.0
Name: clean_risk_score, dtype: float64


In [50]:
# RELOAD - Push fixed dataframe back into SQLite database
df.to_sql('transactions', conn, index=False, if_exists='replace')
print("✅ Database reloaded with fixed risk scores!")

✅ Database reloaded with fixed risk scores!
